# AI Energy Score — CRISP-DM Phase 2: Data Understanding

## Objective

This notebook performs **basic exploratory/descriptive analysis** of the 11 AI Energy Score task files.

At this CRISP-DM stage we are **not modelling yet**. We want to answer:

1. What does each file represent?
2. How many benchmarked models are available for each task?
3. Are there missing values or duplicate records?
4. What are the basic descriptive statistics of GPU energy consumption?
5. Which models are relatively low- or high-energy within each task?
6. Which task files are most relevant to our project on the environmental impact of **generative AI**?

### Important interpretation

Each row is a **model benchmark record for a specific task**, not a random observation from a population.  
Therefore, small task-level sample sizes are acceptable for **descriptive benchmark comparisons**, but we should avoid strong population-level statistical inference.

The main quantitative metric used below is:

**GPU energy consumption in Wh per 1,000 benchmark queries**

The raw CSV variable `total_gpu_energy` is converted as:

\[
	ext{Energy per 1,000 queries} = 	ext{total_gpu_energy} 	imes 1000
\]

Cross-task comparisons are descriptive only because the benchmark workload differs by task.


## 1. What each file represents

The files correspond to different AI tasks. We keep all 11 files for Data Understanding, but we distinguish between:

- **Core generative-AI tasks:** Text Generation, Reasoning, Image Generation, Summarization, Image Captioning
- **Supporting AI tasks:** ASR, Image Classification, Object Detection, Question Answering, Sentence Similarity, Text Classification

This distinction helps us stay aligned with the project title while still using the broader benchmark for context.


In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Change this to Path(".") if the CSV files are in the same folder as the notebook.
DATA_DIR = Path("/mnt/data") if Path("/mnt/data").exists() else Path(".")

FILE_INFO = {
    "asr.csv": ("ASR", "Speech/audio → text", False),
    "image_captioning.csv": ("Image Captioning", "Image → text description", True),
    "image_classification.csv": ("Image Classification", "Image → class/category", False),
    "image_generation.csv": ("Image Generation", "Text prompt → generated image", True),
    "object_detection.csv": ("Object Detection", "Image → detected objects", False),
    "question_answering.csv": ("Question Answering", "Question/context → answer", False),
    "reasoning.csv": ("Reasoning", "Reasoning / multi-step text task", True),
    "sentence_similarity.csv": ("Sentence Similarity", "Text pair → similarity score", False),
    "summarization.csv": ("Summarization", "Long text → shorter generated text", True),
    "text_classification.csv": ("Text Classification", "Text → label/category", False),
    "text_generation.csv": ("Text Generation", "Prompt → generated text", True),
}

pd.set_option("display.max_colwidth", 80)


In [ ]:
# File explanation table
file_explanation = pd.DataFrame(
    [
        {
            "file": file,
            "task": task,
            "what_it_does": description,
            "project_role": "Core GenAI" if core else "Supporting AI"
        }
        for file, (task, description, core) in FILE_INFO.items()
    ]
)

file_explanation


## 2. Load and combine the 11 benchmark files

We preserve the original task label for every row.  
No files are merged on model name; they are only **stacked into one analysis table** so that descriptive statistics can be produced consistently.


In [ ]:
frames = []

for file, (task, description, core) in FILE_INFO.items():
    df_task = pd.read_csv(DATA_DIR / file)
    df_task["task"] = task
    df_task["source_file"] = file
    df_task["core_genai"] = core

    # Convert the raw energy variable to the easier-to-read benchmark unit.
    df_task["energy_wh_per_1000_queries"] = df_task["total_gpu_energy"] * 1000

    frames.append(df_task)

df = pd.concat(frames, ignore_index=True, sort=False)

print("Combined shape:", df.shape)
df.head()


## 3. Dataset overview

For every task we check:

- number of benchmark rows;
- number of unique model names;
- number of missing values;
- number of exact duplicate rows;
- whether a model name appears more than once.

A repeated model name is **not automatically an error**. It may represent repeated benchmark records.  
We only flag it here; the cleaning decision belongs to **Phase 3 — Data Preparation**.


In [ ]:
overview_rows = []

for file, (task, description, core) in FILE_INFO.items():
    x = pd.read_csv(DATA_DIR / file)

    overview_rows.append({
        "task": task,
        "rows": len(x),
        "unique_models": x["model"].nunique(),
        "missing_values": int(x.isna().sum().sum()),
        "exact_duplicate_rows": int(x.duplicated().sum()),
        "repeated_model_records": int(len(x) - x["model"].nunique()),
        "project_role": "Core GenAI" if core else "Supporting AI"
    })

overview = pd.DataFrame(overview_rows)
overview


In [ ]:
print("Total benchmark rows:", len(df))
print("Total missing values:", int(df.isna().sum().sum()))
print("Total exact duplicate rows:", int(df.duplicated().sum()))

# Repeated model names within the same task
repeated_models = (
    df.groupby(["task", "model"])
      .size()
      .reset_index(name="n_records")
      .query("n_records > 1")
      .sort_values(["task", "n_records"], ascending=[True, False])
)

repeated_models


### Interpretation

Small `n` values are acceptable here because the files are **benchmark leaderboards**, not random samples used for causal inference.

For example, if a task has 7 models, our statement should be:

> “Seven models are benchmarked in this task file, and we compare their recorded energy consumption.”

We should **not** claim that those seven models are statistically representative of every AI model in existence.


## 4. Basic descriptive statistics by task

For each task we calculate:

- **n**
- **mean**
- **median**
- **standard deviation**
- **minimum**
- **maximum**

Because energy distributions can be strongly skewed, the **median should always be reported together with the mean**.


In [ ]:
energy_summary = (
    df.groupby("task")["energy_wh_per_1000_queries"]
      .agg(
          n="count",
          mean="mean",
          median="median",
          std="std",
          minimum="min",
          maximum="max"
      )
      .round(2)
      .sort_values("median")
)

energy_summary


In [ ]:
# Add a simple skewness indicator: mean / median.
# A large ratio suggests that high-energy observations are pulling the mean upward.
energy_summary_with_ratio = energy_summary.copy()
energy_summary_with_ratio["mean_to_median_ratio"] = (
    energy_summary_with_ratio["mean"] / energy_summary_with_ratio["median"]
).round(2)

energy_summary_with_ratio


## 5. Visual check: energy distribution across tasks

Energy varies by orders of magnitude across task types, so we use a **logarithmic y-axis**.

This plot is useful for understanding scale and dispersion, but it should **not** be interpreted as a perfectly controlled comparison of task difficulty, because each task has a different benchmark workload.


In [ ]:
task_order = (
    df.groupby("task")["energy_wh_per_1000_queries"]
      .median()
      .sort_values()
      .index
      .tolist()
)

plot_data = [
    df.loc[df["task"] == task, "energy_wh_per_1000_queries"].values
    for task in task_order
]

fig, ax = plt.subplots(figsize=(12, 6))
ax.boxplot(plot_data, labels=task_order, showfliers=True)
ax.set_yscale("log")
ax.set_ylabel("GPU energy (Wh per 1,000 queries, log scale)")
ax.set_xlabel("AI task")
ax.set_title("Distribution of GPU energy across AI Energy Score tasks")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()


## 6. Focus on the five core generative-AI tasks

For the main project, we prioritize:

- Text Generation
- Reasoning
- Image Generation
- Summarization
- Image Captioning

The other files remain useful as broader AI context.


In [ ]:
core_tasks = [
    "Text Generation",
    "Reasoning",
    "Image Generation",
    "Summarization",
    "Image Captioning"
]

core_summary = (
    df[df["task"].isin(core_tasks)]
    .groupby("task")["energy_wh_per_1000_queries"]
    .agg(n="count", mean="mean", median="median", minimum="min", maximum="max")
    .round(2)
    .sort_values("median")
)

core_summary


## 7. Model rankings within each core task

The strongest comparison is **within the same task**.

For each core task, we identify the lowest- and highest-energy benchmarked models.  
This is descriptive: “lowest among the models in this file,” not “best model in the world.”


In [ ]:
def energy_ranking(task, n=5):
    x = (
        df[df["task"] == task][["model", "energy_wh_per_1000_queries", "energy_score"]]
        .sort_values("energy_wh_per_1000_queries")
        .reset_index(drop=True)
    )

    lowest = x.head(n).copy()
    highest = x.tail(n).sort_values("energy_wh_per_1000_queries", ascending=False).copy()

    return lowest, highest

for task in core_tasks:
    low, high = energy_ranking(task, n=5)
    print("\n" + "=" * 80)
    print(task.upper())
    print("\nLowest-energy benchmark records:")
    print(low.to_string(index=False))
    print("\nHighest-energy benchmark records:")
    print(high.to_string(index=False))


## 8. Detailed view: Text Generation

Text Generation has the largest number of benchmark records among the uploaded files, so it is especially useful for our project.

We show the **10 lowest-energy** and **10 highest-energy** benchmark records instead of trying to display all models on one crowded chart.


In [ ]:
text_gen = (
    df[df["task"] == "Text Generation"]
    [["model", "energy_wh_per_1000_queries", "energy_score", "class", "test date"]]
    .sort_values("energy_wh_per_1000_queries")
    .reset_index(drop=True)
)

text_gen.head(10)


In [ ]:
text_gen.tail(10).sort_values("energy_wh_per_1000_queries", ascending=False)


In [ ]:
# Plot the 10 lowest-energy text-generation models
low10 = text_gen.head(10).sort_values("energy_wh_per_1000_queries")

fig, ax = plt.subplots(figsize=(10, 6))
ax.barh(low10["model"], low10["energy_wh_per_1000_queries"])
ax.set_xlabel("GPU energy (Wh per 1,000 queries)")
ax.set_ylabel("Model")
ax.set_title("10 lowest-energy text-generation benchmark records")
plt.tight_layout()
plt.show()


In [ ]:
# Plot the 10 highest-energy text-generation models
high10 = text_gen.tail(10).sort_values("energy_wh_per_1000_queries")

fig, ax = plt.subplots(figsize=(10, 6))
ax.barh(high10["model"], high10["energy_wh_per_1000_queries"])
ax.set_xlabel("GPU energy (Wh per 1,000 queries)")
ax.set_ylabel("Model")
ax.set_title("10 highest-energy text-generation benchmark records")
plt.tight_layout()
plt.show()


## 9. Energy-score distribution

The `energy_score` variable is useful as a compact efficiency label, but our main quantitative variable remains the measured energy value.

Here we simply inspect how the star/score categories are distributed within each task.


In [ ]:
score_distribution = pd.crosstab(df["task"], df["energy_score"])
score_distribution


## 10. Phase 2 conclusions

At the end of **CRISP-DM Phase 2 — Data Understanding**, this dataset should support the following conclusions:

### Data structure
- The project uses **11 task-specific AI Energy Score benchmark files**.
- Each row represents a benchmarked model/task record.
- The files are relatively small, which is acceptable for **descriptive leaderboard analysis**.
- We avoid treating the model lists as random representative samples of all AI models.

### Data quality
- Check the overview table above for missing values and exact duplicate rows.
- Repeated model names, if present, are flagged for later treatment in **Phase 3 — Data Preparation**.

### Main analytical variable
- Our main measure is **GPU energy consumption in Wh per 1,000 queries**.
- Mean and median should both be reported because several task distributions are strongly skewed.

### Project focus
The main generative-AI files are:

1. Text Generation
2. Reasoning
3. Image Generation
4. Summarization
5. Image Captioning

The remaining task files provide broader AI context.

### What this dataset will be used for later
- model-level inference-energy comparisons;
- identifying relatively efficient and energy-intensive models;
- scenario analysis at larger usage scales.

### What this dataset will NOT be used for
- historical forecasting of AI electricity demand;
- causal claims about why one model consumes more energy;
- claiming that the benchmarked models statistically represent all existing AI models.


## 11. Outputs to save for the project

Recommended Phase 2 deliverables:

1. `AI_Energy_Score_dataset_overview.csv`
2. `AI_Energy_Score_descriptive_statistics.csv`
3. Core GenAI model-ranking tables
4. Cross-task energy-distribution figure
5. Text-generation low/high-energy model figures
6. Short Phase 2 findings paragraph for the report / presentation


In [ ]:
# Optional: save the main Phase 2 tables
overview.to_csv(DATA_DIR / "AI_Energy_Score_dataset_overview.csv", index=False)
energy_summary_with_ratio.to_csv(DATA_DIR / "AI_Energy_Score_descriptive_statistics.csv")

print("Saved:")
print(DATA_DIR / "AI_Energy_Score_dataset_overview.csv")
print(DATA_DIR / "AI_Energy_Score_descriptive_statistics.csv")
